# GSE276099 β-caryophyllene: raw-read count reanalysis

**Status:** Method notebook awaiting six checksum-verified NCBI SRA archives, converted FASTQs, and STAR-derived integer counts. This file has no executed results yet.

The contrast is β-caryophyllene versus ethanol vehicle in HepG2 bulk RNA-seq (three GEO libraries per arm). The primary input is the unstranded integer gene-count matrix generated from the six 50-base single-end runs with STAR 2.7.11b and GENCODE human release 48. The [SRA-run manifest](../research/hepg2-bcp-rsem/sra_run_manifest.tsv), [SRA fetch script](../research/hepg2-bcp-rsem/fetch_sra_inputs.py), [conversion script](../research/hepg2-bcp-rsem/convert_sra_to_fastq.py), and [mapping script](../research/hepg2-bcp-rsem/reprocess_star_counts.py) specify source files and processing. The separate ENA FASTQ manifest remains as a source cross-check.

The GEO treatment fields say 150, 151, and 152 µM across replicate labels while the shared protocol says 150 µM. The model below uses treatment only; it does not assume a paired dose or batch design. Three library labels per arm limit dispersion and outlier assessment. Gene-level significance, if found, is within this study only and does not establish a direct molecular target, pathway activity, or cross-context reproducibility.

In [1]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
import pydeseq2

ROOT = Path.cwd()
PROJECT = ROOT / "research/hepg2-bcp-rsem"
RESULTS = PROJECT / "results/raw_counts"
receipt = json.loads((RESULTS / "processing_receipt.json").read_text())
assert receipt["aligner_version"] == "2.7.11b"
assert hashlib.sha256((PROJECT / "reprocess_star_counts.py").read_bytes()).hexdigest() == receipt["pipeline_sha256"]
assert hashlib.sha256((PROJECT / "sra_run_manifest.tsv").read_bytes()).hexdigest() == receipt["run_manifest_sha256"]
assert hashlib.sha256((PROJECT / "sample_manifest.tsv").read_bytes()).hexdigest() == receipt["sample_manifest_sha256"]
assert hashlib.sha256((RESULTS / "conversion_receipt.json").read_bytes()).hexdigest() == receipt["conversion_receipt_sha256"]
for name, expected in receipt["output_sha256"].items():
    observed = hashlib.sha256((RESULTS / name).read_bytes()).hexdigest()
    assert observed == expected, f"Processing receipt mismatch: {name}"
counts = pd.read_csv(RESULTS / "gene_counts_unstranded.tsv.gz", sep="\t", index_col="gene_id")
annotation = pd.read_csv(RESULTS / "gene_annotation.tsv.gz", sep="\t", index_col="gene_id")
star_qc = pd.read_csv(RESULTS / "star_mapping_qc.tsv", sep="\t")
manifest = pd.read_csv(PROJECT / "sra_run_manifest.tsv", sep="\t")
sample_metadata = pd.read_csv(PROJECT / "sample_manifest.tsv", sep="\t", keep_default_na=False)
assert counts.shape[1] == 6 and len(star_qc) == len(manifest) == 6
assert set(counts.columns) == set(manifest.matrix_column) == set(star_qc.matrix_column)
assert set(counts.columns) == set(sample_metadata.matrix_column)
assert not counts.isna().any().any() and (counts >= 0).all().all()
assert np.issubdtype(counts.to_numpy().dtype, np.integer)
print({"genes": len(counts), "libraries": counts.shape[1], "PyDESeq2": pydeseq2.__version__})

{'genes': 78894, 'libraries': 6, 'PyDESeq2': '0.5.4'}


## Sample identity and quality

GEO accessions and source-reported run identities are joined by the pinned manifest. The library-level plots are descriptive QC; they do not create biological replicates. Review mapping percentages and sample clustering before interpreting differential expression.

In [2]:
manifest = manifest.set_index("matrix_column").loc[counts.columns]
sample_metadata = sample_metadata.set_index("matrix_column").loc[counts.columns]
assert (manifest.geo_accession == sample_metadata.geo_accession).all()
assert sample_metadata.arm.value_counts().to_dict() == {"control": 3, "bcp": 3}
condition = np.where(sample_metadata.arm.eq("bcp"), "bcp", "vehicle")
metadata = pd.DataFrame({"condition": pd.Categorical(condition, categories=["vehicle", "bcp"])}, index=counts.columns)
qc = star_qc.set_index("matrix_column").loc[counts.columns].copy()
qc["condition"] = condition
qc["counted_reads"] = counts.sum(axis=0)
qc["detected_genes"] = (counts > 0).sum(axis=0)
for field in ("input_reads", "uniquely_mapped_reads"):
    qc[field] = pd.to_numeric(qc[field], errors="raise")
if qc["input_reads"].le(0).any() or qc["uniquely_mapped_reads"].le(0).any():
    raise ValueError("A library has no input or uniquely mapped reads")
qc.to_csv(RESULTS / "analysis_sample_qc.tsv", sep="\t", index_label="matrix_column")
log_cpm = np.log2(counts.div(counts.sum(axis=0), axis=1) * 1_000_000 + 1)
correlations = log_cpm.corr(method="spearman")
correlations.to_csv(RESULTS / "analysis_sample_correlations.tsv", sep="\t")
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(correlations, vmin=0.8, vmax=1.0, cmap="viridis")
ax.set_xticks(range(6), correlations.columns, rotation=45, ha="right")
ax.set_yticks(range(6), correlations.index)
fig.colorbar(image, ax=ax, label="Spearman correlation of log2(CPM+1)")
fig.tight_layout()
fig.savefig(RESULTS / "analysis_sample_correlations.png", dpi=160)
plt.close(fig)
display(qc[["geo_accession", "run_accession", "condition", "input_reads", "uniquely_mapped_reads", "counted_reads", "detected_genes"]])

,geo_accession,run_accession,condition,input_reads,uniquely_mapped_reads,counted_reads,detected_genes
matrix_column,,,,,,,
read_count_C1,GSM8491080,SRR30495544,vehicle,23457098,20128716,17364362,27358
read_count_C2,GSM8491081,SRR30495543,vehicle,23405054,20082332,17363605,26839
read_count_C3,GSM8491082,SRR30495542,vehicle,23414338,20157768,17413923,26797
read_count_T1,GSM8491083,SRR30495541,bcp,23369393,20119696,17471515,26214
read_count_T2,GSM8491084,SRR30495540,bcp,23388740,20095240,17457733,26342
read_count_T3,GSM8491085,SRR30495539,bcp,23406855,20213877,17588935,26013


## Prespecified count model

Keep a gene if at least three of six libraries have ≥10 unstranded counts. Fit a negative-binomial treatment model with PyDESeq2 0.5.4, testing β-caryophyllene against ethanol vehicle. The table retains all tested genes and reports Benjamini–Hochberg adjusted p values. The count model is exploratory because the experiment has only three labeled libraries per arm, uncertain dose metadata, and no independently verified batch structure.

In [3]:
keep = counts.ge(10).sum(axis=1).ge(3)
model_counts = counts.loc[keep].T.astype(int)
if model_counts.shape[1] < 1000:
    raise ValueError("Too few genes pass the prespecified count filter")
dds = DeseqDataSet(counts=model_counts, metadata=metadata, design="~condition", n_cpus=4)
dds.deseq2()
stats = DeseqStats(dds, contrast=["condition", "bcp", "vehicle"], alpha=0.05, n_cpus=4)
stats.summary()
results = stats.results_df.copy()
results.index.name = "gene_id"
results = results.join(annotation[["gene_name", "gene_type"]], how="left")
results = results.sort_values(["padj", "pvalue"], na_position="last")
results.to_csv(RESULTS / "deseq2_results.tsv", sep="\t")
print({"tested_genes": len(results), "FDR_below_0.05": int(results.padj.lt(0.05).sum()), "FDR_below_0.05_and_abs_log2FC_at_least_1": int((results.padj.lt(0.05) & results.log2FoldChange.abs().ge(1)).sum())})

Using None as control genes, passed at DeseqDataSet initialization


Fitting size factors...
... done in 0.01 seconds.



Fitting dispersions...


... done in 6.16 seconds.

Fitting dispersion trend curve...


... done in 0.73 seconds.

Fitting MAP dispersions...


... done in 7.58 seconds.

Fitting LFCs...


... done in 5.53 seconds.

Calculating cook's distance...
... done in 0.02 seconds.

Replacing 0 outlier genes.

Running Wald tests...


... done in 2.80 seconds.



Log2 fold change & Wald test p-value: condition bcp vs vehicle
                       baseMean  log2FoldChange     lfcSE      stat    pvalue  \
gene_id                                                                         
ENSG00000000419.14  1370.872369        0.065044  0.125759  0.517214  0.605007   
ENSG00000000457.14   136.186264       -0.009417  0.184722 -0.050982  0.959340   
ENSG00000000460.17   514.893011        0.191788  0.124657  1.538521  0.123921   
ENSG00000001036.14  2711.817961       -0.026828  0.102890 -0.260742  0.794291   
ENSG00000001084.13  1373.341019        0.256727  0.119920  2.140828  0.032288   
...                         ...             ...       ...       ...       ...   
ENSG00000310517.1   5210.055462        0.176627  0.098106  1.800379  0.071801   
ENSG00000310521.1     14.743006       -0.238994  0.474679 -0.503486  0.614623   
ENSG00000310526.1     10.446081       -0.240207  0.591335 -0.406211  0.684588   
ENSG00000310560.1   1290.362585        0.25843

{'tested_genes': 15078, 'FDR_below_0.05': 1616, 'FDR_below_0.05_and_abs_log2FC_at_least_1': 240}


## Targeted inspection and limits

PGF, NR4A1, CSF1R, ARTN, and CACNA1E were named before this count reanalysis because they appear in the source paper or earlier processed-matrix audit. Their rows are shown regardless of significance. A nominal pathway or direct target claim requires separate pathway analysis or target engagement; this notebook does not supply either.

In [4]:
targets = ["PGF", "NR4A1", "CSF1R", "ARTN", "CACNA1E"]
target_rows = results[results.gene_name.isin(targets)].copy()
target_rows.to_csv(RESULTS / "prespecified_genes.tsv", sep="\t")
summary = {
    "study": "GSE276099",
    "contrast": "beta-caryophyllene versus ethanol vehicle in HepG2",
    "model": "PyDESeq2 0.5.4 negative binomial ~condition, Wald test, BH FDR",
    "filter": "at least 10 raw unstranded counts in at least 3 of 6 libraries",
    "libraries_per_arm": {"vehicle": 3, "bcp": 3},
    "genes_in_count_matrix": int(len(counts)),
    "tested_genes": int(len(results)),
    "fdr_below_0_05": int(results.padj.lt(0.05).sum()),
    "fdr_below_0_05_abs_log2fc_at_least_1": int((results.padj.lt(0.05) & results.log2FoldChange.abs().ge(1)).sum()),
    "prespecified_gene_symbols": targets,
    "limitations": ["Three labeled libraries per arm", "Dose fields disagree with shared protocol", "Batch and biological independence not independently verified", "No target engagement or cross-study replication"],
}
(RESULTS / "analysis_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
display(target_rows[["gene_name", "baseMean", "log2FoldChange", "pvalue", "padj"]])
print(summary)

,gene_name,baseMean,log2FoldChange,pvalue,padj
gene_id,,,,,
ENSG00000123358.20,NR4A1,1275.820938,0.737151,1.737716e-08,0.000003
ENSG00000182578.14,CSF1R,35.045683,1.615072,5.316987e-05,0.001724
ENSG00000198216.14,CACNA1E,45.334625,-1.177921,5.415555e-04,0.009722
ENSG00000117407.17,ARTN,115.405177,-0.673615,1.235469e-03,0.017480
ENSG00000119630.14,PGF,25.319141,-1.174437,2.657488e-03,0.029605


{'study': 'GSE276099', 'contrast': 'beta-caryophyllene versus ethanol vehicle in HepG2', 'model': 'PyDESeq2 0.5.4 negative binomial ~condition, Wald test, BH FDR', 'filter': 'at least 10 raw unstranded counts in at least 3 of 6 libraries', 'libraries_per_arm': {'vehicle': 3, 'bcp': 3}, 'genes_in_count_matrix': 78894, 'tested_genes': 15078, 'fdr_below_0_05': 1616, 'fdr_below_0_05_abs_log2fc_at_least_1': 240, 'prespecified_gene_symbols': ['PGF', 'NR4A1', 'CSF1R', 'ARTN', 'CACNA1E'], 'limitations': ['Three labeled libraries per arm', 'Dose fields disagree with shared protocol', 'Batch and biological independence not independently verified', 'No target engagement or cross-study replication']}
